# 🤖 Workshop: Treinamento de Robô Quadrúpede com NVIDIA Isaac Lab

## Módulo 00 — Setup do Ambiente Cloud ☁️
**Duração:** 25 minutos | **Summit de IA — Joinville, SC, Brasil**

---

### 👋 Bem-vindos!

Neste workshop de **2h30min** você vai sair do zero e ter um robô quadrúpede andando dentro de uma simulação física de alta fidelidade, treinado com Reinforcement Learning.

### 🏗️ O que vamos construir:

```
☁️  GPU Cloud  →  🦾 Isaac Sim  →  🧠 Treinar PPO  →  🐾 Anymal C andando!
```

### 📅 Agenda Visual

| # | Módulo | Tópico | Tempo |
|---|--------|--------|-------|
| 00 | **Setup** | Ambiente cloud + Isaac Sim | 0:00 – 0:25 |
| 01 | **Isaac Lab** | Framework e arquitetura | 0:25 – 0:55 |
| 02 | **Quadrúpede** | Ambiente MDP do Anymal C | 0:55 – 1:25 |
| 03 | **Treinamento** | RSL-RL + PPO | 1:25 – 2:00 |
| 04 | **Resultados** | Análise + próximos passos | 2:00 – 2:30 |

### 🎯 Objetivo deste módulo
Ter um ambiente GPU funcional com Isaac Sim rodando e pronto para executar nossos scripts de treinamento.


## 💻 Hardware Necessário

Isaac Sim é pesado! Você precisa de uma GPU dedicada. Veja as opções:

| GPU | VRAM | Performance | Custo/hora | Plataforma |
|-----|------|-------------|------------|------------|
| RTX 3090 | 24 GB | ⭐⭐⭐⭐ | ~$0.30–0.50 | Vast.ai |
| RTX 4090 | 24 GB | ⭐⭐⭐⭐⭐ | ~$0.50–0.80 | Vast.ai / Brev |
| A100 40GB | 40 GB | ⭐⭐⭐⭐⭐ | ~$1.50–2.00 | Vast.ai / Brev |
| A10G | 24 GB | ⭐⭐⭐⭐ | ~$0.80–1.20 | NVIDIA Brev |
| Local RTX 30/40 | 8 GB+ | ⭐⭐⭐ | Energia elétrica | Sua máquina |

> ⚠️ **Requisito mínimo:** GPU com **8 GB VRAM** e CUDA 11.8+. Para o workshop usaremos **RTX 3090 ou 4090**.

> 💡 **Dica de custo:** Para o workshop completo (~3h) você gastará aproximadamente **R$3 a R$8** numa RTX 4090 no Vast.ai.


## ☁️ Opção 1: Vast.ai — Passo a Passo

O Vast.ai é um marketplace de GPUs onde pessoas físicas e empresas alugam suas placas. É geralmente a opção **mais barata**.

### 📋 Passos:

**1. Criar conta**
- Acesse [vast.ai](https://vast.ai) e clique em **Sign Up**
- Use seu e-mail e crie uma senha

**2. Adicionar crédito**
- Vá em **Billing** → **Add Credit**
- Adicione pelo menos **$5 USD** (suficiente para o workshop)
- Aceita cartão de crédito internacional

**3. Buscar template Isaac Sim**
- Clique em **Search** (ícone de lupa)
- No campo **Image**, busque: `nvcr.io/nvidia/isaac-sim:5.1.0`

**4. Selecionar GPU**
- Filtre por: `RTX 3090` ou `RTX 4090`
- Verifique: **VRAM ≥ 24GB**, **RAM ≥ 32GB**, **Disco ≥ 60GB**

**5. Configurar portas (OBRIGATÓRIO para streaming)**
```
Portas a expor: 47995-48012, 49100, 8888, 6006
```

**6. Conectar via SSH**
```bash
ssh -p <PORTA> root@<IP> -L 8888:localhost:8888 -L 6006:localhost:6006
```


In [ ]:
# ✅ VERIFICAÇÃO 1: GPU disponível?
!nvidia-smi
import os, sys
print(f'\n🐍 Python: {sys.version}')
print(f'📁 Disco:')
!df -h / | tail -1


## ☁️ Opção 2: NVIDIA Brev — Passo a Passo

O NVIDIA Brev é a plataforma oficial da NVIDIA para desenvolvimento em GPU na nuvem.

### 📋 Passos:

**1. Criar conta:** [brev.nvidia.com](https://brev.nvidia.com)

**2. Instalar o CLI do Brev**
```bash
# macOS:
brew install brevdev/homebrew-brev/brev

# Linux:
curl -fsSL https://raw.githubusercontent.com/brevdev/brev-cli/main/bin/install-brev.sh | bash
```

**3. Login e criar instância GPU**
```bash
brev login
brev create isaac-workshop --gpu A10G --image nvcr.io/nvidia/isaac-sim:5.1.0
brev open isaac-workshop  # abre no VS Code
```

> 💡 O Brev oferece **créditos gratuitos** para novos usuários NVIDIA!


## 🐳 Iniciando o Container Isaac Sim

```bash
# Pull da imagem (~15 GB)
docker pull nvcr.io/nvidia/isaac-sim:5.1.0

# Rodar container com streaming
docker run --gpus all --rm \
  -e ACCEPT_EULA=Y -e PRIVACY_CONSENT=Y \
  -p 47995-48012:47995-48012 -p 49100:49100 \
  -p 8888:8888 -p 6006:6006 \
  -v ~/workshop:/workshop \
  -it nvcr.io/nvidia/isaac-sim:5.1.0 bash
```

```bash
# Dentro do container — instalar Isaac Lab
git clone https://github.com/isaac-sim/IsaacLab.git /IsaacLab
cd /IsaacLab && ./isaaclab.sh --install && ./isaaclab.sh --install rsl_rl
```


In [ ]:
# ✅ VERIFICAÇÃO 2: Isaac Sim e Isaac Lab instalados?
import os

caminhos = {
    'Isaac Sim': ['/isaac-sim', '/opt/isaac-sim'],
    'Isaac Lab': ['/IsaacLab', os.environ.get('ISAACLAB_PATH', '')],
}

for nome, paths in caminhos.items():
    encontrado = any(p and os.path.exists(p) for p in paths)
    status = '✅' if encontrado else '❌'
    print(f'{status} {nome}')

for var in ['ISAACLAB_PATH', 'PYTHONPATH', 'CUDA_HOME']:
    print(f'  {var} = {os.environ.get(var, "(não definida)")}')


In [ ]:
# ✅ VERIFICAÇÃO FINAL: PyTorch + CUDA
import torch
print(f'🔥 PyTorch: {torch.__version__}')
print(f'🖥️  CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'🎯 GPU: {torch.cuda.get_device_name(0)}')
    print(f'💾 VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')
    x = torch.randn(1000, 1000, device='cuda')
    y = torch.mm(x, x.T)
    print(f'✅ Teste matmul GPU: shape={y.shape}')


In [ ]:
# ✅ VERIFICAÇÃO: Pacote workshop_quadrupede instalado?
import os
try:
    import workshop_quadrupede
    print('✅ workshop_quadrupede instalado!')
except ImportError:
    print('❌ Não instalado. Execute:')
    print('   cd /IsaacLabTutorial && /isaac-sim/python.sh -m pip install -e workshop/source/workshop_quadrupede')


## ✅ Setup Completo! Parabéns! 🎉

### 📋 Checklist Final

- [ ] GPU detectada com `nvidia-smi`
- [ ] CUDA disponível via PyTorch
- [ ] Isaac Sim em `/isaac-sim`
- [ ] Isaac Lab instalado
- [ ] `workshop_quadrupede` instalado
- [ ] Portas 49100 e 47995-48012 abertas

### ❓ Problemas comuns:

| Problema | Solução |
|----------|---------| 
| `CUDA not available` | Verificar `--gpus all` no docker run |
| `ImportError: torch` | Usar `/isaac-sim/python.sh` |
| Streaming não conecta | Verificar portas no painel Vast.ai/Brev |
| `No module workshop_quadrupede` | `pip install -e workshop/source/workshop_quadrupede` |

---

### ➡️ Próximo: Módulo 01 — Introdução ao Isaac Lab
